# Data Engineering - Preparation des Donnees E-commerce

## Pipeline de nettoyage et transformation

Ce notebook implémente un pipeline de data engineering structuré pour préparer les données RetailRocket.

### Étapes du pipeline :
1. **Chargement** - Import des données brutes
2. **Audit qualité** - Contrôle initial des données
3. **Nettoyage** - Gestion des valeurs manquantes et doublons
4. **Transformation** - Conversion des types
5. **Feature Engineering** - Création de variables dérivées
6. **Validation** - Contrôle qualité final
7. **Export** - Sauvegarde des données nettoyées

---

## 1. Configuration et Imports

In [ ]:
# =============================================================================
# CONFIGURATION DU PIPELINE
# =============================================================================

import pandas as pd
import numpy as np
from datetime import datetime
import warnings
import os

# Suppression des warnings pour un output propre
warnings.filterwarnings('ignore')

# Configuration pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Chemins des fichiers
RAW_DATA_PATH = 'data/'
CLEAN_DATA_PATH = 'outputs/data/'

# Créer le dossier de sortie si nécessaire
os.makedirs(CLEAN_DATA_PATH, exist_ok=True)

print("[OK] Configuration terminee")
print(f"   Donnees brutes : {RAW_DATA_PATH}")
print(f"   Donnees nettoyees : {CLEAN_DATA_PATH}")

## 2. Chargement des Données Brutes

In [ ]:
# =============================================================================
# FONCTION DE CHARGEMENT AVEC LOGGING
# =============================================================================

def load_data(file_path, nrows=None):
    """
    Charge un fichier CSV avec logging des informations.
    
    Args:
        file_path (str): Chemin du fichier CSV
        nrows (int, optional): Nombre de lignes à charger
    
    Returns:
        pd.DataFrame: DataFrame chargé
    """
    print(f"Chargement de {file_path}...")
    
    df = pd.read_csv(file_path, nrows=nrows)
    
    print(f"   [OK] {len(df):,} lignes x {df.shape[1]} colonnes")
    print(f"   Memoire : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    return df

# Chargement des datasets
print("CHARGEMENT DES DONNEES BRUTES")
print("=" * 50)

# Dataset principal : événements utilisateurs
events_raw = load_data(f'{RAW_DATA_PATH}events.csv')

# Hiérarchie des catégories
categories_raw = load_data(f'{RAW_DATA_PATH}category_tree.csv')

# Propriétés des produits (échantillon pour optimiser la mémoire)
item_props_raw = load_data(f'{RAW_DATA_PATH}item_properties_part1.csv', nrows=500_000)

print("\n[OK] Tous les fichiers charges avec succes!")

## 3. Audit Qualité Initial

In [ ]:
# =============================================================================
# FONCTION D'AUDIT QUALITÉ
# =============================================================================

def audit_dataframe(df, name="DataFrame"):
    """
    Effectue un audit qualité complet d'un DataFrame.
    
    Args:
        df (pd.DataFrame): DataFrame à auditer
        name (str): Nom du DataFrame pour l'affichage
    
    Returns:
        dict: Dictionnaire avec les métriques d'audit
    """
    print(f"\nAUDIT QUALITE : {name}")
    print("=" * 60)
    
    audit = {
        'name': name,
        'rows': len(df),
        'columns': df.shape[1],
        'memory_mb': df.memory_usage(deep=True).sum() / 1024**2,
        'duplicates': df.duplicated().sum(),
        'missing_total': df.isnull().sum().sum(),
        'column_details': {}
    }
    
    print(f"\nDimensions : {audit['rows']:,} lignes x {audit['columns']} colonnes")
    print(f"Memoire : {audit['memory_mb']:.2f} MB")
    print(f"Doublons : {audit['duplicates']:,} ({audit['duplicates']/len(df)*100:.2f}%)")
    
    # Analyse par colonne
    print("\nDetail par colonne :")
    print("-" * 60)
    print(f"{'Colonne':<20} {'Type':<12} {'Manquants':<12} {'Uniques':<12}")
    print("-" * 60)
    
    for col in df.columns:
        dtype = str(df[col].dtype)
        missing = df[col].isnull().sum()
        missing_pct = (missing / len(df)) * 100
        unique = df[col].nunique()
        
        audit['column_details'][col] = {
            'dtype': dtype,
            'missing': missing,
            'missing_pct': missing_pct,
            'unique': unique
        }
        
        missing_str = f"{missing:,} ({missing_pct:.1f}%)"
        print(f"{col:<20} {dtype:<12} {missing_str:<12} {unique:,}")
    
    return audit

# Audit des données brutes
audit_events = audit_dataframe(events_raw, "EVENTS")
audit_categories = audit_dataframe(categories_raw, "CATEGORIES")
audit_items = audit_dataframe(item_props_raw, "ITEM_PROPERTIES")

## 4. Nettoyage des Données

In [ ]:
# =============================================================================
# CLASSE DE NETTOYAGE DES DONNÉES
# =============================================================================

class DataCleaner:
    """
    Classe pour nettoyer les données e-commerce.
    Implémente les bonnes pratiques de data engineering.
    """
    
    def __init__(self, df, name="DataFrame"):
        self.df = df.copy()
        self.name = name
        self.log = []
        self.initial_rows = len(df)
        
    def _log(self, message):
        """Ajoute un message au log et l'affiche."""
        timestamp = datetime.now().strftime("%H:%M:%S")
        log_entry = f"[{timestamp}] {message}"
        self.log.append(log_entry)
        print(f"   {message}")
    
    def remove_duplicates(self, subset=None):
        """
        Supprime les lignes dupliquées.
        
        Args:
            subset (list, optional): Colonnes à considérer pour les doublons
        """
        before = len(self.df)
        self.df = self.df.drop_duplicates(subset=subset)
        removed = before - len(self.df)
        self._log(f"Doublons supprimes : {removed:,}")
        return self
    
    def handle_missing(self, column, strategy='drop', fill_value=None):
        """
        Gère les valeurs manquantes pour une colonne.
        
        Args:
            column (str): Nom de la colonne
            strategy (str): 'drop', 'fill', 'fill_median', 'fill_mode', 'keep'
            fill_value: Valeur de remplacement si strategy='fill'
        """
        missing_before = self.df[column].isnull().sum()
        
        if missing_before == 0:
            self._log(f"[OK] {column} : Aucune valeur manquante")
            return self
        
        if strategy == 'drop':
            self.df = self.df.dropna(subset=[column])
            self._log(f"[DROP] {column} : {missing_before:,} lignes supprimees (NaN)")
            
        elif strategy == 'fill':
            self.df[column] = self.df[column].fillna(fill_value)
            self._log(f"[FILL] {column} : {missing_before:,} NaN remplaces par {fill_value}")
            
        elif strategy == 'fill_median':
            median = self.df[column].median()
            self.df[column] = self.df[column].fillna(median)
            self._log(f"[FILL] {column} : {missing_before:,} NaN remplaces par mediane ({median:.2f})")
            
        elif strategy == 'fill_mode':
            mode = self.df[column].mode()[0]
            self.df[column] = self.df[column].fillna(mode)
            self._log(f"[FILL] {column} : {missing_before:,} NaN remplaces par mode ({mode})")
            
        elif strategy == 'keep':
            self._log(f"[KEEP] {column} : {missing_before:,} NaN conserves (comportement attendu)")
        
        return self
    
    def filter_outliers(self, column, method='iqr', threshold=1.5):
        """
        Filtre les valeurs aberrantes.
        
        Args:
            column (str): Nom de la colonne
            method (str): 'iqr' ou 'zscore'
            threshold (float): Seuil pour la détection
        """
        before = len(self.df)
        
        if method == 'iqr':
            Q1 = self.df[column].quantile(0.25)
            Q3 = self.df[column].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - threshold * IQR
            upper = Q3 + threshold * IQR
            self.df = self.df[(self.df[column] >= lower) & (self.df[column] <= upper)]
            
        elif method == 'zscore':
            z_scores = np.abs((self.df[column] - self.df[column].mean()) / self.df[column].std())
            self.df = self.df[z_scores <= threshold]
        
        removed = before - len(self.df)
        self._log(f"[FILTER] {column} : {removed:,} outliers supprimes (methode {method})")
        return self
    
    def validate_values(self, column, valid_values):
        """
        Valide que les valeurs sont dans une liste autorisée.
        
        Args:
            column (str): Nom de la colonne
            valid_values (list): Liste des valeurs autorisées
        """
        before = len(self.df)
        invalid = ~self.df[column].isin(valid_values)
        invalid_count = invalid.sum()
        
        if invalid_count > 0:
            self.df = self.df[~invalid]
            self._log(f"[WARN] {column} : {invalid_count:,} valeurs invalides supprimees")
        else:
            self._log(f"[OK] {column} : Toutes les valeurs sont valides")
        
        return self
    
    def get_clean_data(self):
        """Retourne le DataFrame nettoyé."""
        final_rows = len(self.df)
        removed_total = self.initial_rows - final_rows
        print(f"\nResume : {removed_total:,} lignes supprimees ({removed_total/self.initial_rows*100:.2f}%)")
        print(f"   Lignes initiales : {self.initial_rows:,}")
        print(f"   Lignes finales   : {final_rows:,}")
        return self.df

print("[OK] Classe DataCleaner definie")

In [ ]:
# =============================================================================
# NETTOYAGE DU DATASET EVENTS
# =============================================================================

print("NETTOYAGE DU DATASET EVENTS")
print("=" * 50)

# Types d'événements valides
VALID_EVENTS = ['view', 'addtocart', 'transaction']

# Pipeline de nettoyage
cleaner = DataCleaner(events_raw, "events")

events_clean = (
    cleaner
    .remove_duplicates()  # Supprimer les doublons exacts
    .validate_values('event', VALID_EVENTS)  # Valider les types d'événements
    .handle_missing('timestamp', strategy='drop')  # Timestamp obligatoire
    .handle_missing('visitorid', strategy='drop')  # Visitor obligatoire
    .handle_missing('itemid', strategy='drop')  # Item obligatoire
    .handle_missing('transactionid', strategy='keep')  # NaN attendu pour view/cart
    .get_clean_data()
)

## 5. Conversion des Types

In [ ]:
# =============================================================================
# CONVERSION DES TYPES DE DONNÉES
# =============================================================================

def convert_types(df):
    """
    Convertit les types de données pour optimiser la mémoire et l'analyse.
    
    Args:
        df (pd.DataFrame): DataFrame à convertir
    
    Returns:
        pd.DataFrame: DataFrame avec types convertis
    """
    print("CONVERSION DES TYPES")
    print("=" * 50)
    
    df = df.copy()
    memory_before = df.memory_usage(deep=True).sum() / 1024**2
    
    # --- 1. Conversion du timestamp en datetime ---
    print("\nConversion timestamp -> datetime")
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
    print(f"   Periode : {df['datetime'].min()} -> {df['datetime'].max()}")
    
    # --- 2. Conversion des IDs en types appropriés ---
    print("\nOptimisation des types numeriques")
    
    # visitorid et itemid : garder comme int64 (valeurs grandes)
    # mais on peut utiliser uint32 si les valeurs le permettent
    if df['visitorid'].max() < 2**32:
        df['visitorid'] = df['visitorid'].astype('uint32')
        print("   visitorid : int64 → uint32")
    
    if df['itemid'].max() < 2**32:
        df['itemid'] = df['itemid'].astype('uint32')
        print("   itemid : int64 → uint32")
    
    # --- 3. Event comme catégorie (économise beaucoup de mémoire) ---
    print("\nConversion en categories")
    df['event'] = df['event'].astype('category')
    print(f"   event : object → category ({df['event'].cat.categories.tolist()})")
    
    # --- 4. transactionid : Int64 nullable ---
    # Permet de garder les NaN avec un type entier
    df['transactionid'] = df['transactionid'].astype('Int64')  # Nullable integer
    print("   transactionid : float64 → Int64 (nullable)")
    
    # Calcul du gain mémoire
    memory_after = df.memory_usage(deep=True).sum() / 1024**2
    savings = ((memory_before - memory_after) / memory_before) * 100
    
    print(f"\nOptimisation memoire :")
    print(f"   Avant  : {memory_before:.2f} MB")
    print(f"   Après  : {memory_after:.2f} MB")
    print(f"   Gain   : {savings:.1f}%")
    
    return df

# Appliquer les conversions
events_typed = convert_types(events_clean)

# Vérifier les types finaux
print("\nTypes finaux :")
print(events_typed.dtypes)

## 6. Feature Engineering

In [ ]:
# =============================================================================
# CRÉATION DE FEATURES TEMPORELLES
# =============================================================================

def create_temporal_features(df):
    """
    Crée des features temporelles à partir de la colonne datetime.
    
    Args:
        df (pd.DataFrame): DataFrame avec colonne 'datetime'
    
    Returns:
        pd.DataFrame: DataFrame enrichi
    """
    print("CREATION DES FEATURES TEMPORELLES")
    print("=" * 50)
    
    df = df.copy()
    
    # Extraction des composantes temporelles
    df['date'] = df['datetime'].dt.date
    df['year'] = df['datetime'].dt.year.astype('uint16')
    df['month'] = df['datetime'].dt.month.astype('uint8')
    df['day'] = df['datetime'].dt.day.astype('uint8')
    df['hour'] = df['datetime'].dt.hour.astype('uint8')
    df['minute'] = df['datetime'].dt.minute.astype('uint8')
    df['day_of_week'] = df['datetime'].dt.dayofweek.astype('uint8')  # 0=Lundi
    df['day_name'] = df['datetime'].dt.day_name()
    df['week_of_year'] = df['datetime'].dt.isocalendar().week.astype('uint8')
    
    # Features dérivées
    df['is_weekend'] = (df['day_of_week'] >= 5).astype('uint8')
    df['is_business_hour'] = ((df['hour'] >= 9) & (df['hour'] <= 18)).astype('uint8')
    
    # Période de la journée
    def get_day_period(hour):
        if 6 <= hour < 12:
            return 'morning'
        elif 12 <= hour < 14:
            return 'lunch'
        elif 14 <= hour < 18:
            return 'afternoon'
        elif 18 <= hour < 22:
            return 'evening'
        else:
            return 'night'
    
    df['day_period'] = df['hour'].apply(get_day_period).astype('category')
    
    print("   [OK] date, year, month, day, hour, minute")
    print("   [OK] day_of_week, day_name, week_of_year")
    print("   [OK] is_weekend, is_business_hour")
    print("   [OK] day_period (morning/lunch/afternoon/evening/night)")
    
    return df

# Appliquer les features temporelles
events_featured = create_temporal_features(events_typed)

In [ ]:
# =============================================================================
# CRÉATION DE FEATURES COMPORTEMENTALES
# =============================================================================

def create_behavioral_features(df):
    """
    Crée des features comportementales agrégées.
    
    Args:
        df (pd.DataFrame): DataFrame des événements
    
    Returns:
        tuple: (events_df, visitor_features, product_features)
    """
    print("\nCREATION DES FEATURES COMPORTEMENTALES")
    print("=" * 50)
    
    # --- Features par visiteur ---
    print("\n> Features par visiteur :")
    
    visitor_features = df.groupby('visitorid').agg(
        total_events=('event', 'count'),
        unique_items=('itemid', 'nunique'),
        first_event=('datetime', 'min'),
        last_event=('datetime', 'max'),
        unique_days=('date', 'nunique')
    ).reset_index()
    
    # Compter les événements par type
    event_counts = df.pivot_table(
        index='visitorid', 
        columns='event', 
        aggfunc='size', 
        fill_value=0
    ).reset_index()
    event_counts.columns = ['visitorid', 'n_addtocart', 'n_transaction', 'n_view']
    
    visitor_features = visitor_features.merge(event_counts, on='visitorid')
    
    # Calculer des ratios
    visitor_features['cart_rate'] = (
        visitor_features['n_addtocart'] / visitor_features['n_view']
    ).fillna(0)
    
    visitor_features['conversion_rate'] = (
        visitor_features['n_transaction'] / visitor_features['n_view']
    ).fillna(0)
    
    # Durée d'engagement
    visitor_features['engagement_days'] = (
        (visitor_features['last_event'] - visitor_features['first_event']).dt.days
    )
    
    # Segmentation visiteur
    def segment_visitor(row):
        if row['n_transaction'] > 0:
            return 'buyer'
        elif row['n_addtocart'] > 0:
            return 'cart_abandoner'
        else:
            return 'browser'
    
    visitor_features['segment'] = visitor_features.apply(segment_visitor, axis=1)
    visitor_features['segment'] = visitor_features['segment'].astype('category')
    
    print(f"   [OK] {len(visitor_features):,} visiteurs profiles")
    print(f"   Features : {list(visitor_features.columns)}")
    
    # --- Features par produit ---
    print("\n> Features par produit :")
    
    product_features = df.groupby('itemid').agg(
        total_events=('event', 'count'),
        unique_visitors=('visitorid', 'nunique'),
        first_seen=('datetime', 'min'),
        last_seen=('datetime', 'max')
    ).reset_index()
    
    # Compter par type d'événement
    product_events = df.pivot_table(
        index='itemid', 
        columns='event', 
        aggfunc='size', 
        fill_value=0
    ).reset_index()
    product_events.columns = ['itemid', 'n_addtocart', 'n_transaction', 'n_view']
    
    product_features = product_features.merge(product_events, on='itemid')
    
    # Taux de conversion produit
    product_features['view_to_cart'] = (
        product_features['n_addtocart'] / product_features['n_view']
    ).fillna(0)
    
    product_features['cart_to_purchase'] = (
        product_features['n_transaction'] / product_features['n_addtocart']
    ).replace([np.inf, -np.inf], 0).fillna(0)
    
    product_features['overall_conversion'] = (
        product_features['n_transaction'] / product_features['n_view']
    ).fillna(0)
    
    print(f"   [OK] {len(product_features):,} produits profiles")
    print(f"   Features : {list(product_features.columns)}")
    
    return df, visitor_features, product_features

# Créer les features comportementales
events_final, visitor_features, product_features = create_behavioral_features(events_featured)

In [ ]:
# =============================================================================
# CRÉATION D'UN FLAG CONVERSION
# =============================================================================

print("CREATION DU FLAG CONVERSION")
print("=" * 50)

# Identifier les visiteurs qui ont acheté
buyers = events_final[events_final['event'] == 'transaction']['visitorid'].unique()

# Ajouter un flag sur les événements
events_final['visitor_converted'] = events_final['visitorid'].isin(buyers).astype('uint8')

print(f"   Visiteurs acheteurs : {len(buyers):,}")
print(f"   Événements de convertis : {events_final['visitor_converted'].sum():,}")

## 7. Contrôle Qualité Final

In [ ]:
# =============================================================================
# VALIDATION FINALE DES DONNÉES
# =============================================================================

def validate_data(df, name="DataFrame"):
    """
    Effectue des validations de qualité sur les données finales.
    
    Args:
        df (pd.DataFrame): DataFrame à valider
        name (str): Nom du DataFrame
    
    Returns:
        bool: True si toutes les validations passent
    """
    print(f"\nVALIDATION FINALE : {name}")
    print("=" * 60)
    
    all_passed = True
    
    # Test 1 : Pas de doublons
    duplicates = df.duplicated().sum()
    status = "[PASS]" if duplicates == 0 else "[FAIL]"
    print(f"   {status} - Doublons : {duplicates:,}")
    if duplicates > 0:
        all_passed = False
    
    # Test 2 : Colonnes critiques sans NaN
    critical_cols = ['timestamp', 'visitorid', 'event', 'itemid']
    for col in critical_cols:
        if col in df.columns:
            missing = df[col].isnull().sum()
            status = "[PASS]" if missing == 0 else "[FAIL]"
            print(f"   {status} - {col} sans NaN : {missing:,}")
            if missing > 0:
                all_passed = False
    
    # Test 3 : Types d'événements valides
    if 'event' in df.columns:
        valid_events = {'view', 'addtocart', 'transaction'}
        actual_events = set(df['event'].unique())
        is_valid = actual_events.issubset(valid_events)
        status = "[PASS]" if is_valid else "[FAIL]"
        print(f"   {status} - Types d'événements valides : {actual_events}")
        if not is_valid:
            all_passed = False
    
    # Test 4 : Timestamps dans une plage raisonnable
    if 'datetime' in df.columns:
        min_date = df['datetime'].min()
        max_date = df['datetime'].max()
        is_valid = min_date.year >= 2000 and max_date.year <= 2030
        status = "[PASS]" if is_valid else "[FAIL]"
        print(f"   {status} - Dates valides : {min_date.date()} -> {max_date.date()}")
        if not is_valid:
            all_passed = False
    
    # Test 5 : IDs positifs
    for col in ['visitorid', 'itemid']:
        if col in df.columns:
            min_val = df[col].min()
            is_valid = min_val >= 0
            status = "[PASS]" if is_valid else "[FAIL]"
            print(f"   {status} - {col} positifs : min={min_val}")
            if not is_valid:
                all_passed = False
    
    # Résumé
    print("\n" + "-" * 60)
    if all_passed:
        print("TOUTES LES VALIDATIONS PASSEES !")
    else:
        print("[WARN] CERTAINES VALIDATIONS ONT ECHOUE")
    
    return all_passed

# Valider les datasets
validate_data(events_final, "EVENTS_FINAL")
validate_data(visitor_features, "VISITOR_FEATURES")
validate_data(product_features, "PRODUCT_FEATURES")

In [ ]:
# =============================================================================
# RÉSUMÉ DU PIPELINE
# =============================================================================

print("\n" + "=" * 70)
print("RESUME DU PIPELINE DATA ENGINEERING")
print("=" * 70)

print("\n> EVENTS (donnees principales) :")
print(f"   Lignes initiales : {len(events_raw):,}")
print(f"   Lignes finales   : {len(events_final):,}")
print(f"   Colonnes         : {events_final.shape[1]}")
print(f"   Memoire          : {events_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n> VISITOR_FEATURES :")
print(f"   Visiteurs profilés : {len(visitor_features):,}")
print(f"   Features creees    : {visitor_features.shape[1]}")

print("\n> PRODUCT_FEATURES :")
print(f"   Produits profilés : {len(product_features):,}")
print(f"   Features creees   : {product_features.shape[1]}")

print("\n> NOUVELLES FEATURES :")
new_features = [
    'datetime', 'date', 'year', 'month', 'day', 'hour', 'minute',
    'day_of_week', 'day_name', 'week_of_year',
    'is_weekend', 'is_business_hour', 'day_period',
    'visitor_converted'
]
for feat in new_features:
    if feat in events_final.columns:
        print(f"   [OK] {feat}")

print("\n" + "=" * 70)

## 8. Export des Données Nettoyées

In [ ]:
# =============================================================================
# EXPORT DES DONNÉES PROPRES
# =============================================================================

print("EXPORT DES DONNEES NETTOYEES")
print("=" * 50)

# Events nettoyés (sans la colonne datetime pour le CSV - problème de format)
events_export = events_final.copy()
events_export['datetime_str'] = events_export['datetime'].astype(str)
events_export = events_export.drop(columns=['datetime', 'date'])

events_export.to_csv(f'{CLEAN_DATA_PATH}events_clean.csv', index=False)
print(f"[OK] {CLEAN_DATA_PATH}events_clean.csv")

# Visitor features
visitor_export = visitor_features.copy()
visitor_export['first_event'] = visitor_export['first_event'].astype(str)
visitor_export['last_event'] = visitor_export['last_event'].astype(str)
visitor_export.to_csv(f'{CLEAN_DATA_PATH}visitor_features.csv', index=False)
print(f"[OK] {CLEAN_DATA_PATH}visitor_features.csv")

# Product features
product_export = product_features.copy()
product_export['first_seen'] = product_export['first_seen'].astype(str)
product_export['last_seen'] = product_export['last_seen'].astype(str)
product_export.to_csv(f'{CLEAN_DATA_PATH}product_features.csv', index=False)
print(f"[OK] {CLEAN_DATA_PATH}product_features.csv")

# Export format Parquet (plus efficace pour les gros volumes)
try:
    events_final.to_parquet(f'{CLEAN_DATA_PATH}events_clean.parquet', index=False)
    print(f"[OK] {CLEAN_DATA_PATH}events_clean.parquet")
except Exception as e:
    print(f"[WARN] Export Parquet non disponible : {e}")

print("\nPipeline termine avec succes!")

In [ ]:
# =============================================================================
# APERÇU DES DONNÉES FINALES
# =============================================================================

print("APERCU DES DONNEES FINALES")
print("=" * 50)

print("\n> Events (5 premieres lignes) :")
display(events_final.head())

print("\n> Visitor Features (5 premieres lignes) :")
display(visitor_features.head())

print("\n> Product Features (5 premieres lignes) :")
display(product_features.head())